# Draftly Legal Corpus Processing Pipeline

This notebook keeps the original exploratory case-index stages and finishes
with the production, file-based legal corpus under `data/processed/`.

The canonical retrieval store contains normalized Markdown for statutes,
amendments, gazettes, institution guides, and case law; provenance records;
case records; deterministic topic tables; section-specific citation edges; an
unresolved-citation queue; and a lawyer-verification sample.

No database is required. Generated legal-authority links remain `unverified`
until a lawyer signs them off.

**Stages**

0. Validate inputs and build the complete discovery index.
1. Select usable conveyancing text and audit extraction quality.
2. Extract exploratory statute mentions and nearby section references.
3. Attach curated conveyancing topics.
4. Write exploratory case-index records and quality summaries.
5. Create a blank, reviewer-owned retrieval-evaluation template.
6. Build and verify the canonical `data/processed/` retrieval store.

## Setup and configuration

Run from the repository root or from `notebooks/`. Set `QUICK_RUN = True`
while developing; the normal setting processes every usable conveyancing
judgment.

In [1]:
import json
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (
            candidate / "data/legal-sources/manifests"
        ).exists():
            return candidate
    raise FileNotFoundError("Could not locate the Draftly repository root.")


ROOT = find_repo_root(Path.cwd())
MANIFESTS = ROOT / "data/legal-sources/manifests"
CASE_LAW = ROOT / "data/legal-sources/library/case-law"
OUTPUT = ROOT / "data/processed/legal-case-index"
OUTPUT.mkdir(parents=True, exist_ok=True)

QUICK_RUN = False
SAMPLE_LIMIT = 300 if QUICK_RUN else None
MIN_TEXT_CHARS = 500
CONVERT_HTML = False

print("Repository:", ROOT)
print("Mode:", "quick sample" if QUICK_RUN else "full corpus")
print("Output:", OUTPUT)

Repository: D:\projects\draftly
Mode: full corpus
Output: D:\projects\draftly\data\processed\legal-case-index


## Input validation

The manifests are contracts. Failing early on missing files or columns is safer
than producing a plausible-looking but incomplete index.

In [2]:
def read_manifest(filename: str, required_columns: set[str]) -> pd.DataFrame:
    path = MANIFESTS / filename
    if not path.exists():
        raise FileNotFoundError(f"Required manifest is missing: {path}")

    frame = pd.read_csv(path, dtype=str).fillna("")
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{filename} is missing columns: {sorted(missing)}")
    return frame


common_index = read_manifest(
    "case-law-commonlii-index.csv",
    {"db", "year", "case_no", "citation", "title", "url"},
)
common_matches = read_manifest(
    "case-law-commonlii-conveyancing.csv",
    {"db", "year", "case_no", "hits", "chars", "text_file"},
)
court_index = read_manifest(
    "case-law-courts.csv",
    {"court", "file", "url", "chars", "hits", "conveyancing", "needs_ocr", "text_file"},
)
slr_index = read_manifest(
    "slr-modern-cases.csv",
    {"year", "volume", "case_id", "title", "case_url", "status"},
)
source_registry = read_manifest(
    "source-registry.csv",
    {"source_id", "official_title", "act_or_ordinance_no", "year", "source_type"},
)
topic_source_map = read_manifest(
    "topic-sources.csv",
    {"topic_id", "slug", "source_id"},
)

print(
    {
        "commonlii_index": len(common_index),
        "commonlii_conveyancing_text": len(common_matches),
        "official_courts": len(court_index),
        "modern_slr_references": len(slr_index),
        "legal_sources": len(source_registry),
        "topic_source_edges": len(topic_source_map),
    }
)

{'commonlii_index': 9539, 'commonlii_conveyancing_text': 3703, 'official_courts': 5474, 'modern_slr_references': 630, 'legal_sources': 90, 'topic_source_edges': 174}


## Stage 0 - Build the complete discovery index

All source records remain visible. A CommonLII citation-index row is enriched
with its conveyancing text record when one exists. Official-court years inferred
from filenames are explicitly marked as inferred. Modern SLR references stay
`indexed_reference` because their full report text is not locally available.

Records are not merged merely because party names look similar. That crosswalk
requires a verified matching stage.

In [3]:
YEAR_RX = re.compile(r"(?<!\d)(?:18|19|20)\d{2}(?!\d)")


def to_int(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0).astype(int)


def path_exists(relative_path: str) -> bool:
    return bool(relative_path) and (ROOT / relative_path).is_file()


def infer_year(value: str) -> str:
    years = YEAR_RX.findall(value)
    return years[-1] if years else ""


def normalize_title(value: str) -> str:
    value = unicodedata.normalize("NFKC", value or "")
    value = re.sub(r"\s+-\s+(?:NLR|SLR)\s+-.*$", "", value, flags=re.I)
    value = re.sub(r"\[[12]\d{3}\].*$", "", value)
    value = re.sub(r"\b(?:versus|vs\.?|v\.)\b", " v ", value, flags=re.I)
    value = re.sub(r"[^a-z0-9]+", " ", value.casefold())
    return re.sub(r"\s+", " ", value).strip()


def commonlii_records() -> pd.DataFrame:
    keys = ["db", "year", "case_no"]
    enrichment = common_matches[
        keys + ["hits", "chars", "text_file"]
    ].drop_duplicates(keys)
    frame = common_index.merge(enrichment, how="left", on=keys).fillna("")
    text_available = frame["text_file"].map(path_exists)
    return pd.DataFrame(
        {
            "case_id": "commonlii-" + frame["db"] + "-" + frame["year"] + "-" + frame["case_no"],
            "source": "commonlii",
            "source_record_id": frame["db"] + "/" + frame["year"] + "/" + frame["case_no"],
            "court": frame["db"].str.upper(),
            "year": frame["year"],
            "year_status": "reported",
            "citation": frame["citation"],
            "title": frame["title"],
            "url": frame["url"],
            "text_path": frame["text_file"],
            "status": text_available.map({True: "full_text", False: "indexed_reference"}),
            "conveyancing_match": frame["text_file"].ne(""),
            "needs_ocr": False,
            "chars": to_int(frame["chars"]),
            "hits": to_int(frame["hits"]),
            "text_available": text_available,
        }
    )


def court_records() -> pd.DataFrame:
    frame = court_index.copy()
    years = frame["file"].map(infer_year)
    text_available = frame["text_file"].map(path_exists)
    needs_ocr = frame["needs_ocr"].eq("1")
    status = pd.Series("full_text", index=frame.index)
    status = status.mask(~text_available, "missing_text")
    status = status.mask(needs_ocr, "needs_ocr")
    return pd.DataFrame(
        {
            "case_id": "courts-" + frame["court"] + "-" + frame["file"],
            "source": "official_courts",
            "source_record_id": frame["court"] + "/" + frame["file"],
            "court": frame["court"].str.upper(),
            "year": years,
            "year_status": years.ne("").map({True: "filename_inferred", False: "unknown"}),
            "citation": "",
            "title": frame["file"].map(
                lambda name: re.sub(r"[-_.]+", " ", re.sub(r"\.pdf$", "", name, flags=re.I)).strip()
            ),
            "url": frame["url"],
            "text_path": frame["text_file"],
            "status": status,
            "conveyancing_match": frame["conveyancing"].eq("1"),
            "needs_ocr": needs_ocr,
            "chars": to_int(frame["chars"]),
            "hits": to_int(frame["hits"]),
            "text_available": text_available,
        }
    )


def modern_slr_records() -> pd.DataFrame:
    frame = slr_index.copy()
    return pd.DataFrame(
        {
            "case_id": "lawlanka-slr-" + frame["case_id"],
            "source": "lawlanka_slr_index",
            "source_record_id": frame["case_id"],
            "court": "",
            "year": frame["year"],
            "year_status": "reported",
            "citation": "",
            "title": frame["title"],
            "url": frame["case_url"],
            "text_path": "",
            "status": frame["status"].replace("", "indexed_reference"),
            "conveyancing_match": False,
            "needs_ocr": False,
            "chars": 0,
            "hits": 0,
            "text_available": False,
        }
    )


master = pd.concat(
    [commonlii_records(), court_records(), modern_slr_records()],
    ignore_index=True,
)
master["normalized_title"] = master["title"].map(normalize_title)
master["normalized_citation"] = (
    master["citation"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
)
master["duplicate_citation"] = (
    master["normalized_citation"].ne("")
    & master.duplicated("normalized_citation", keep=False)
)

if master["case_id"].duplicated().any():
    duplicates = master.loc[master["case_id"].duplicated(False), "case_id"].tolist()
    raise ValueError(f"Non-unique case IDs: {duplicates[:10]}")

master = master.sort_values(["source", "year", "case_id"], kind="stable").reset_index(drop=True)
master.to_csv(OUTPUT / "master_case_index.csv", index=False)

source_summary = (
    master.groupby("source", dropna=False)
    .agg(
        records=("case_id", "size"),
        conveyancing_matches=("conveyancing_match", "sum"),
        local_text=("text_available", "sum"),
        needs_ocr=("needs_ocr", "sum"),
    )
    .reset_index()
)
source_summary.to_csv(OUTPUT / "source_summary.csv", index=False)

duplicate_citations = master[
    master["duplicate_citation"]
].sort_values(["normalized_citation", "source"])
duplicate_citations.to_csv(OUTPUT / "duplicate_citations.csv", index=False)

print("Master records:", len(master))
display(source_summary)

Master records: 15643


,source,records,conveyancing_matches,local_text,needs_ocr
0,commonlii,9539,3656,3656,0
1,lawlanka_slr_index,630,0,0,0
2,official_courts,5474,1418,5474,1


## Stage 1 - Select text and audit extraction quality

Only locally available, conveyancing-matched, non-OCR-backlog judgments enter
the text-processing corpus. Cleaning preserves paragraph boundaries. Encoding
damage is flagged for review rather than silently rewritten.

In [4]:
def load_text(relative_path: str) -> str:
    path = ROOT / relative_path
    if not path.is_file():
        return ""
    return path.read_text(encoding="utf-8", errors="replace")


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.replace("\x00", " ").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\bPage\s+\d+\s+of\s+\d+\b", " ", text, flags=re.I)
    text = re.sub(r"[ \t]+", " ", text)
    lines = [line.strip() for line in text.splitlines()]
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def text_quality_flags(raw: str, cleaned: str) -> list[str]:
    flags = []
    if not cleaned:
        flags.append("empty")
    if len(cleaned) < MIN_TEXT_CHARS:
        flags.append("too_short")
    if "\ufffd" in raw:
        flags.append("replacement_character")
    if any(marker in raw for marker in ("â€", "â€™", "Ã", "Â")):
        flags.append("possible_mojibake")
    return flags


processing_cases = master[
    master["conveyancing_match"]
    & master["text_available"]
    & ~master["needs_ocr"]
].copy()
processing_cases = processing_cases.sort_values(
    ["source", "court", "year", "case_id"], kind="stable"
)
if SAMPLE_LIMIT is not None:
    processing_cases = processing_cases.head(SAMPLE_LIMIT).copy()

text_cache: dict[str, str] = {}
quality_rows = []
for row in processing_cases.itertuples(index=False):
    raw = load_text(row.text_path)
    cleaned = clean_text(raw)
    text_cache[row.case_id] = cleaned
    flags = text_quality_flags(raw, cleaned)
    quality_rows.append(
        {
            "case_id": row.case_id,
            "source": row.source,
            "text_path": row.text_path,
            "manifest_chars": row.chars,
            "raw_chars": len(raw),
            "clean_chars": len(cleaned),
            "replacement_characters": raw.count("\ufffd"),
            "quality_flags": ";".join(flags),
            "usable": not flags,
        }
    )

text_quality = pd.DataFrame(quality_rows)
text_quality.to_csv(OUTPUT / "case_text_quality.csv", index=False)
quality_by_flag = (
    text_quality.assign(
        quality_flag=text_quality["quality_flags"].replace("", "ok")
    )
    .groupby("quality_flag")
    .size()
    .rename("cases")
    .reset_index()
)

print("Cases selected for text processing:", len(processing_cases))
print("Clean text characters:", sum(map(len, text_cache.values())))
display(quality_by_flag)

Cases selected for text processing: 5074
Clean text characters: 93010361


,quality_flag,cases
0,ok,4185
1,possible_mojibake,1
2,replacement_character,888


### Optional HTML normalization

The public LawLanka volume pages are discovery aids, not judgment text. Their
CSV indexes are the pipeline input. Set `CONVERT_HTML = True` only when local
Markdown copies of those pages are useful for inspection.

In [5]:
if CONVERT_HTML:
    from markdownify import markdownify

    html_output = OUTPUT / "html-markdown"
    html_files = sorted(CASE_LAW.rglob("*.html"))
    for source_path in html_files:
        destination = (
            html_output / source_path.relative_to(CASE_LAW)
        ).with_suffix(".md")
        destination.parent.mkdir(parents=True, exist_ok=True)
        html = source_path.read_text(encoding="utf-8", errors="replace")
        destination.write_text(
            markdownify(html, heading_style="ATX").strip(),
            encoding="utf-8",
        )
    print(f"Converted {len(html_files)} HTML files to {html_output}")
else:
    print("HTML conversion skipped.")

HTML conversion skipped.


## Stage 2 - Extract statute and nearby section mentions

The matcher uses only registry rows classified as statutes or amendments.
Base-law titles can match by title. An amendment is linked only when its title
and identifying number/year appear nearby; this avoids assigning one generic
"Amendment Act" mention to several different amendments.

Section references are collected only from a bounded window around each statute
mention. They are candidate links for legal review, not verified holdings.

In [6]:
LAW_TYPES = {"statute", "amendment"}
SECTION_GROUP_RX = re.compile(
    r"\b(?:sections?|secs?\.?|ss?\.?|s\.)\s*"
    r"((?:\d+[A-Za-z]?"
    r"(?:\s*(?:,|and|or|&|to|-)\s*\d+[A-Za-z]?)*))",
    flags=re.I,
)


def title_pattern(title: str) -> re.Pattern:
    tokens = re.findall(r"[A-Za-z0-9]+", unicodedata.normalize("NFKC", title))
    if not tokens:
        raise ValueError(f"Cannot build a title matcher for {title!r}")
    return re.compile(r"\b" + r"[^A-Za-z0-9]+".join(map(re.escape, tokens)) + r"\b", re.I)


def section_sort_key(section: str) -> tuple[int, str]:
    match = re.fullmatch(r"(\d+)([A-Za-z]?)", section)
    return (int(match.group(1)), match.group(2).casefold()) if match else (10**9, section)


def sections_near(text: str, start: int, end: int, radius: int = 450) -> list[str]:
    window = text[max(0, start - radius) : min(len(text), end + radius)]
    sections = {
        number
        for group in SECTION_GROUP_RX.findall(window)
        for number in re.findall(r"\d+[A-Za-z]?", group)
    }
    return sorted(sections, key=section_sort_key)


law_rows = source_registry[
    source_registry["source_type"].isin(LAW_TYPES)
].copy()

law_matchers = []
for row in law_rows.itertuples(index=False):
    number_year_pattern = None
    if row.source_type == "amendment":
        if not row.act_or_ordinance_no or not row.year or row.year == "0":
            continue
        number_year_pattern = re.compile(
            rf"\b(?:No\.?\s*)?{re.escape(row.act_or_ordinance_no)}\s+of\s+"
            rf"{re.escape(row.year)}\b",
            re.I,
        )
    law_matchers.append(
        {
            "source_id": row.source_id,
            "title": row.official_title,
            "source_type": row.source_type,
            "title_rx": title_pattern(row.official_title),
            "number_year_rx": number_year_pattern,
        }
    )

edge_rows = []
for case in processing_cases.itertuples(index=False):
    text = text_cache[case.case_id]
    for law in law_matchers:
        accepted_mentions = []
        for mention in law["title_rx"].finditer(text):
            if law["number_year_rx"] is not None:
                identity_window = text[
                    max(0, mention.start() - 180) : min(len(text), mention.end() + 180)
                ]
                if not law["number_year_rx"].search(identity_window):
                    continue
            accepted_mentions.append(mention)

        if not accepted_mentions:
            continue

        sections = {
            section
            for mention in accepted_mentions
            for section in sections_near(text, mention.start(), mention.end())
        }
        edge_rows.append(
            {
                "case_id": case.case_id,
                "source_id": law["source_id"],
                "source_title": law["title"],
                "source_type": law["source_type"],
                "mention_count": len(accepted_mentions),
                "sections": ";".join(sorted(sections, key=section_sort_key)),
                "match_method": (
                    "title_and_number_year"
                    if law["number_year_rx"] is not None
                    else "exact_title"
                ),
                "review_status": "candidate",
            }
        )

edge_columns = [
    "case_id",
    "source_id",
    "source_title",
    "source_type",
    "mention_count",
    "sections",
    "match_method",
    "review_status",
]
case_statute_links = pd.DataFrame(edge_rows, columns=edge_columns)
case_statute_links.to_csv(OUTPUT / "case_statute_links.csv", index=False)

print("Law-title matchers:", len(law_matchers))
print("Case-statute candidate edges:", len(case_statute_links))
if not case_statute_links.empty:
    display(
        case_statute_links["source_id"]
        .value_counts()
        .head(15)
        .rename_axis("source_id")
        .reset_index(name="cases")
    )

Law-title matchers: 74
Case-statute candidate edges: 4438


,source_id,cases
0,SRC030,1623
1,SRC029,587
2,SRC071,504
3,SRC027,267
4,SRC059,220
5,SRC001,185
6,SRC028,131
7,SRC005,127
8,SRC049,112
9,SRC014,88


## Stage 3 - Attach curated topics

Topic assignments come from `topic-sources.csv`, the normalized source of
truth. The notebook no longer reparses Markdown manifests with a broad
`SRC\d+` regex.

In [7]:
known_source_ids = set(source_registry["source_id"])
unknown_topic_sources = sorted(set(topic_source_map["source_id"]) - known_source_ids)
if unknown_topic_sources:
    raise ValueError(
        f"topic-sources.csv references unknown source IDs: {unknown_topic_sources}"
    )

source_to_topics = (
    topic_source_map.groupby("source_id")["topic_id"]
    .apply(lambda values: sorted(set(values)))
    .to_dict()
)

case_topic_rows = []
for edge in case_statute_links.itertuples(index=False):
    for topic_id in source_to_topics.get(edge.source_id, []):
        case_topic_rows.append(
            {
                "case_id": edge.case_id,
                "topic_id": topic_id,
                "source_id": edge.source_id,
                "assignment_method": "statute_to_curated_topic",
                "review_status": "candidate",
            }
        )

case_topic_links = (
    pd.DataFrame(
        case_topic_rows,
        columns=[
            "case_id",
            "topic_id",
            "source_id",
            "assignment_method",
            "review_status",
        ],
    )
    .drop_duplicates()
    .sort_values(["case_id", "topic_id", "source_id"], kind="stable")
)
case_topic_links.to_csv(OUTPUT / "case_topic_links.csv", index=False)

print("Case-topic candidate edges:", len(case_topic_links))
print("Cases with at least one topic:", case_topic_links["case_id"].nunique())

Case-topic candidate edges: 10527
Cases with at least one topic: 3023


## Stage 4 - Write retrieval records and pipeline summary

Each JSONL row retains source provenance and local text location. Candidate
statute and topic links are nested, while the raw judgment text stays in its
source file to avoid duplicating the corpus.

In [8]:
quality_by_case = text_quality.set_index("case_id")["quality_flags"].to_dict()

statutes_by_case = {}
for case_id, group in case_statute_links.groupby("case_id"):
    statutes_by_case[case_id] = [
        {
            "source_id": row.source_id,
            "sections": row.sections.split(";") if row.sections else [],
            "mention_count": int(row.mention_count),
            "match_method": row.match_method,
            "review_status": row.review_status,
        }
        for row in group.itertuples(index=False)
    ]

topics_by_case = (
    case_topic_links.groupby("case_id")["topic_id"]
    .apply(lambda values: sorted(set(values)))
    .to_dict()
    if not case_topic_links.empty
    else {}
)

records_path = OUTPUT / "cases.jsonl"
with records_path.open("w", encoding="utf-8") as handle:
    for case in processing_cases.itertuples(index=False):
        statute_links = statutes_by_case.get(case.case_id, [])
        record = {
            "case_id": case.case_id,
            "source": case.source,
            "source_record_id": case.source_record_id,
            "court": case.court,
            "year": case.year,
            "year_status": case.year_status,
            "citation": case.citation,
            "title": case.title,
            "url": case.url,
            "text_path": case.text_path,
            "chars": int(case.chars),
            "hits": int(case.hits),
            "quality_flags": [
                flag
                for flag in quality_by_case.get(case.case_id, "").split(";")
                if flag
            ],
            "statute_links": statute_links,
            "statutes_cited": [link["source_id"] for link in statute_links],
            "topics": topics_by_case.get(case.case_id, []),
            "link_status": "candidate",
        }
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "quick_run": QUICK_RUN,
    "sample_limit": SAMPLE_LIMIT,
    "master_records": int(len(master)),
    "processing_records": int(len(processing_cases)),
    "records_with_quality_flags": int(text_quality["quality_flags"].ne("").sum()),
    "candidate_case_statute_links": int(len(case_statute_links)),
    "candidate_case_topic_links": int(len(case_topic_links)),
    "cases_with_statute_links": int(case_statute_links["case_id"].nunique()),
    "cases_with_topic_links": int(case_topic_links["case_id"].nunique()),
}
(OUTPUT / "pipeline_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("Retrieval records:", records_path)

{
  "generated_at_utc": "2026-07-16T09:56:26.118447+00:00",
  "quick_run": false,
  "sample_limit": null,
  "master_records": 15643,
  "processing_records": 5074,
  "records_with_quality_flags": 889,
  "candidate_case_statute_links": 4438,
  "candidate_case_topic_links": 10527,
  "cases_with_statute_links": 3023,
  "cases_with_topic_links": 3023
}
Retrieval records: D:\projects\draftly\data\processed\legal-case-index\cases.jsonl


## Stage 5 - Reviewer-owned retrieval evaluation set

A gold evaluation set must be authored or approved by a lawyer. The pipeline
therefore creates only the schema and never overwrites an existing reviewed
file. Empty expected fields are not treated as verified answers.

In [9]:
evaluation_path = OUTPUT / "retrieval_eval_gold.csv"
evaluation_columns = [
    "question_id",
    "question",
    "workflow_step",
    "expected_source_ids",
    "expected_sections",
    "expected_case_ids",
    "review_status",
    "reviewer_notes",
]

if not evaluation_path.exists():
    pd.DataFrame(columns=evaluation_columns).to_csv(evaluation_path, index=False)
    print("Created blank evaluation template:", evaluation_path)
else:
    existing_eval = pd.read_csv(evaluation_path, dtype=str).fillna("")
    missing = set(evaluation_columns) - set(existing_eval.columns)
    if missing:
        raise ValueError(
            f"Existing evaluation file is missing columns: {sorted(missing)}"
        )
    print("Preserved existing evaluation file:", evaluation_path)

print("Gold questions:", len(pd.read_csv(evaluation_path)))

Created blank evaluation template: D:\projects\draftly\data\processed\legal-case-index\retrieval_eval_gold.csv
Gold questions: 0


## What to improve next

The normalized corpus and deterministic section-link pass are complete. The
next work is review and retrieval evaluation:

1. Have a lawyer complete `verification-sample.csv` and calculate precision by
   confidence method before treating links as retrieval authority.
2. Prioritize `unresolved_citations.csv` by repeated passage, source, and
   conveyancing topic; add deterministic aliases before considering a bounded
   model-assisted review pass.
3. Segment authoritative statute Markdown into stable section nodes and connect
   verified case edges to those nodes.
4. Curate historical statute short names and spelling variants to improve recall
   without weakening deterministic provenance.
5. Build a lawyer-approved retrieval question set and report precision@k and
   recall@k for the topic-to-statute-to-case walk.
6. Continue repairing flagged encoding damage while retaining original text and
   conversion provenance.

## Stage 6 - Canonical normalized retrieval store

The exploratory output above remains useful for analysis. Production retrieval
uses only `data/processed/`. The scripts below repair the CommonLII manifest
join, normalize every available source into Markdown, create section-specific
case links, preserve unresolved mentions, and generate a 30-edge lawyer review
sheet. OCR is deliberately not invoked from the notebook because cloud use must
remain an explicit operator decision.

In [ ]:
import subprocess
import sys

PRODUCTION = ROOT / "data/processed"
required_outputs = [
    "documents.csv",
    "cases.jsonl",
    "topics.csv",
    "topic-sources.csv",
    "topics.json",
    "case_statute_section_links.csv",
    "unresolved_citations.csv",
    "verification-sample.csv",
]

REBUILD_PRODUCTION_STORE = not all(
    (PRODUCTION / filename).exists() for filename in required_outputs
)

if REBUILD_PRODUCTION_STORE:
    pending_ocr = pd.read_csv(MANIFESTS / "conversion-registry.csv", dtype=str).fillna("")
    pending_ocr = pending_ocr[
        ~pending_ocr["status"].isin(["converted", "fallback", "fallback-pdftotext"])
    ]
    if not pending_ocr.empty:
        raise RuntimeError(
            "OCR outputs are incomplete. Run scripts/convert_to_text.py explicitly "
            "before rebuilding the production store."
        )
    commands = [
        [sys.executable, str(ROOT / "scripts/repair_commonlii_index.py")],
        [sys.executable, str(ROOT / "scripts/build_processed_store.py"), "--require-complete"],
        [sys.executable, str(ROOT / "scripts/build_case_citation_links.py"), "--write-unresolved"],
        [sys.executable, str(ROOT / "scripts/build_verification_sample.py")],
    ]
    for command in commands:
        subprocess.run(command, cwd=ROOT, check=True)

missing_outputs = [
    filename for filename in required_outputs
    if not (PRODUCTION / filename).is_file()
]
if missing_outputs:
    raise FileNotFoundError(f"Missing production outputs: {missing_outputs}")

documents = pd.read_csv(PRODUCTION / "documents.csv", dtype=str).fillna("")
links = pd.read_csv(PRODUCTION / "case_statute_section_links.csv", dtype=str).fillna("")
unresolved = pd.read_csv(PRODUCTION / "unresolved_citations.csv", dtype=str).fillna("")
verification = pd.read_csv(PRODUCTION / "verification-sample.csv", dtype=str).fillna("")
with (PRODUCTION / "cases.jsonl").open(encoding="utf-8") as handle:
    production_cases = [json.loads(line) for line in handle if line.strip()]

document_ids = set(documents["doc_id"])
assert documents["doc_id"].is_unique
assert len(production_cases) == len({case["case_id"] for case in production_cases})
assert all(case["doc_id"] in document_ids for case in production_cases)
assert all((ROOT / path).is_file() for path in documents["text_path"])
assert len(documents[documents["source"] == "commonlii"]) == len(common_matches)
assert links["section"].ne("").all() and links["evidence_snippet"].ne("").all()
assert links["review_status"].eq("unverified").all()
assert len(verification) == 30
assert verification["extraction_status"].eq("unverified").all()

production_summary = {
    "documents": len(documents),
    "cases": len(production_cases),
    "commonlii_conveyancing_joined": len(documents[documents["source"] == "commonlii"]),
    "section_links": len(links),
    "unresolved_mentions": len(unresolved),
    "lawyer_verification_rows": len(verification),
}
print(json.dumps(production_summary, indent=2))